In [1]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/akanksha/formulacode/datasmith
# %cd /mnt/sdd1/atharvas/formulacode/datasmith/

import datetime

import pandas as pd

from datasmith.logging_config import get_logger

logger = get_logger("notebooks.building_reports_pr")

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/akanksha/formulacode/datasmith


/mnt/sdd1/akanksha/formulacode/datasmith/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
commits_df = pd.read_parquet(
    "/mnt/sdd1/atharvas/formulacode/datasmith/scratch/artifacts/pipeflush/merge_commits_filtered_with_patch.parquet"
)
print(commits_df.columns)

Index(['sha', 'date', 'message', 'total_additions', 'total_deletions',
       'total_files_changed', 'files_changed', 'patch', 'has_asv',
       'file_change_summary',
       ...
       'pr_base_topics', 'pr_base_trees_url', 'pr_base_updated_at',
       'pr_base_url', 'pr_base_visibility', 'pr_base_watchers',
       'pr_base_watchers_count', 'pr_base_web_commit_signoff_required',
       'pr_base_sha', 'container_name'],
      dtype='object', length=145)


In [ ]:
# from datasmith.execution.collect_commits import collect_merge_shas

# commits = collect_merge_shas(repo="astropy/astropy")

Paginating GitHub:   0%|          | 0/100 [00:00<?, ?page/s]

Paginating GitHub:  38%|███▊      | 38/100 [00:00<00:00, 122.43page/s]
15:29:17 INFO     datasmith: Collected 3141 merged PR SHAs (non-null) from astropy/astropy.


In [3]:
small_df = commits_df.head(10)

for _, row in small_df.iterrows():
    print(row["pr_url"])

https://api.github.com/repos/not522/ac-library-python/pulls/70
https://api.github.com/repos/not522/ac-library-python/pulls/55
https://api.github.com/repos/not522/ac-library-python/pulls/69
https://api.github.com/repos/not522/ac-library-python/pulls/61
https://api.github.com/repos/not522/ac-library-python/pulls/58
https://api.github.com/repos/not522/ac-library-python/pulls/56
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/92
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/133
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/238
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/258


In [4]:
small_df.columns

Index(['sha', 'date', 'message', 'total_additions', 'total_deletions',
       'total_files_changed', 'files_changed', 'patch', 'has_asv',
       'file_change_summary',
       ...
       'pr_base_topics', 'pr_base_trees_url', 'pr_base_updated_at',
       'pr_base_url', 'pr_base_visibility', 'pr_base_watchers',
       'pr_base_watchers_count', 'pr_base_web_commit_signoff_required',
       'pr_base_sha', 'container_name'],
      dtype='object', length=145)

In [5]:
small_df.iloc[5]

sha                                             b098e9866da8a640af02640c474128fb1526f4b6
date                                                           2021-11-19T09:33:26+09:00
message                                Merge pull request #56 from masa-aa/patch-1\n\...
total_additions                                                                        1
total_deletions                                                                        2
                                                             ...                        
pr_base_watchers                                                                     229
pr_base_watchers_count                                                               229
pr_base_web_commit_signoff_required                                                False
pr_base_sha                                     a30b7e590271d7b77459946695ae8ce984e50f0a
container_name                         not522-ac-library-python-b098e9866da8a640af026...
Name: 5, Length: 145,

In [6]:
from tqdm.auto import tqdm

from datasmith.scrape.build_pr_report import build_pr_report

reports = []


# with ThreadPoolExecutor(max_workers=100) as executor:
#     futures = {
#         executor.submit(
#             build_pr_report,
#             link=commit['url'],
#             summarize_llm=False,
#             add_classification=False,
#         ): commit for commit in commits[:750]
#     }
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         commit = futures[future]
#         try:
#             report = future.result()
#             reports.append(report)
#         except Exception as e:
#             logger.error(f"Error processing commit {commit['url']}: {e}")
# above as a for loop:
h_map = {}
for _, pr in tqdm(small_df.iterrows()):
    if not pr["pr_url"]:
        reports.append(None)
        continue
    print(pr["pr_url"])
    report, prob_statement, hints, classification, difficulty, performance_issue = build_pr_report(
        link=pr["pr_url"],
        summarize_llm=True,
        add_classification=True,
        patch=pr["patch"],
    )
    h_map[pr["pr_url"]] = (report, prob_statement, hints, classification, difficulty, performance_issue)
    reports.append(report)

/mnt/sdd1/akanksha/formulacode/datasmith/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
07:11:12 WARNING  simple_useragent.core: Falling back to historic user agent.
0it [00:00, ?it/s]

https://api.github.com/repos/not522/ac-library-python/pulls/70
ISSUE_THREAD_keys dict_keys(['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason'])


1it [00:06,  6.24s/it]

https://api.github.com/repos/not522/ac-library-python/pulls/55
https://api.github.com/repos/not522/ac-library-python/pulls/69
ISSUE_THREAD_keys dict_keys(['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason'])


3it [00:07,  2.13s/it]

https://api.github.com/repos/not522/ac-library-python/pulls/61
ISSUE_THREAD_keys dict_keys(['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason'])


8it [00:09,  1.50it/s]

https://api.github.com/repos/not522/ac-library-python/pulls/58
https://api.github.com/repos/not522/ac-library-python/pulls/56
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/92
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/133
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/238
ISSUE_THREAD_keys dict_keys(['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignee', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'author_association', 'type', 'active_lock_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason'])
https://api.github.com/repos/xarray-contrib/xbatcher/pulls/258
ISSUE_THREAD_keys dict_keys(['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'use

10it [00:19,  2.00s/it]


In [56]:
report, prob_statement, hints, classification, difficulty, performance_issue = build_pr_report(
    link="https://api.github.com/repos/xarray-contrib/xbatcher/pulls/112",
    summarize_llm=True,
    add_classification=False,
    patch="",
)
print(prob_statement)

## Problem
Generating batches in `__init__` is slow and memory intensive, related to [ISSUE_NUM] and [ISSUE_NUM]. Initialization is changed to load indices into memory rather than corresponding datasets. The change enabled the initialization of a 1.7 TB dataset with 1m+ samples, 30+ spatial features, in about 10 seconds which was previously overloading memory.

Rather than filling `_batches` with `DataArrays` and `Datasets`, it is filled with indices from `selector = {key: slice for key, slice in zip(dims, slices)}`. This required an update to the `concat_input_dims` option where the operation is done in `__getitem__()`. It is possible that this change decreases performance when this `concat_input_dims=True`.

## Related Issues
The issue is related to the following issues:
- Issue 0: Generate batches lazily
- Issue 1: Cache batches

### Issue 0: Generate batches lazily
This issue discusses the need to generate batches lazily to improve performance. A similar implementation was made in 

In [10]:
from pprint import pprint

pprint(h_map)

{'https://api.github.com/repos/not522/ac-library-python/pulls/55': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/56': ('NOT_A_VALID_PR',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    '',
                                                                    False),
 'https://api.github.com/repos/not522/ac-library-python/pulls/58': ('NOT_A_VALID_PR',
              

In [11]:
h_map["https://api.github.com/repos/not522/ac-library-python/pulls/70"]

('\n### Hints\n\n\nThe code has been reviewed and appears to be good to go, with no issues or concerns raised.\n\n\n### Performance Issue\n\n{"label": "YES", "reason": "Implementing pow_mod as a wrapper of pow", "confidence": 80, "flags": ["mentions-speed", "startup"]}\n\n\n### LLM Generated summary\n\n### Problem\nThe issue is about implementing the `atcoder.math.pow_mod` function as a wrapper of the built-in `pow` function without introducing additional assertions.\n\n### Background\nThe discussion started with a comment on Issue 67, which suggested adding `pow_mod` in `math.py`. It was noted that Python\'s built-in `pow` function has a `mod` argument, and it was proposed to introduce `atcoder.math.pow_mod` as a wrapper for consistency with the original ACL.\n\n### Proposed Implementation\nThe user wants to implement the wrapper and has two questions:\n1. Do we need additional assertions in the wrapper?\n2. Do we still need to add tests for that?\n\n### Analysis\nThe original impleme

In [8]:
count = sum(1 for v in h_map.values() if v[0] not in ("NOT_A_PERFORMANCE_COMMIT", "NOT_A_VALID_PR"))
print(count)

1


In [ ]:
# df = pd.DataFrame([{**c, **{"report": report}} for c, report in zip(commits, reports)])

In [73]:
# df[["url", "report"]][~df["report"].str.contains("NOT_A_VALID_PR")].url.values

In [ ]:
# logger.info(f"PR Report:\n{report}")
print(report)

NOT_A_VALID_PR
